In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/santander-customer-transaction-prediction/train.csv')
X = df.drop(["target", "ID_code"], axis=1)
y = df["target"]

In [2]:
from sklearn.model_selection import StratifiedKFold, train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import numpy as np

test_df = pd.read_csv('../data/raw/santander-customer-transaction-prediction/test.csv')
X_test = test_df.drop("ID_code", axis=1)


def run_cv(num_leaves):

    test_preds = np.zeros(len(X_test))
    oof_preds = np.zeros(len(X))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_auc_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(
            n_estimators=1000,
            learning_rate=0.05,
            random_state=42,
            num_leaves=num_leaves
        )
        
        model.fit(
            X_train_fold, y_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(stopping_rounds=100),
                lgb.log_evaluation(period=100)
            ]
        )
        
        fold_preds = model.predict_proba(X_val_fold)[:, 1]
        fold_auc = roc_auc_score(y_val_fold, fold_preds)
        fold_auc_scores.append(fold_auc)

        oof_preds[val_idx] = fold_preds
        test_preds += (
            model.predict_proba(
                X_test,
                num_iteration=model.best_iteration_
            )[:, 1]
            / skf.n_splits
        )
        
        print(f"Fold {fold + 1} AUC: {fold_auc:.6f}")
        oof_auc = roc_auc_score(y, oof_preds)

    return {
        "num_leaves": num_leaves,
        "mean_auc": np.mean(fold_auc_scores),
        "oof_auc": oof_auc,
        "std_auc": np.std(fold_auc_scores)
    }

In [7]:
results = []

for leaves in [8, 16, 31, 64, 128]:
    results.append(run_cv(leaves))

[LightGBM] [Info] Number of positive: 16079, number of negative: 143921
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.063828 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51000
[LightGBM] [Info] Number of data points in the train set: 160000, number of used features: 200
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.100494 -> initscore=-2.191750
[LightGBM] [Info] Start training from score -2.191750
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.822509	valid_0's binary_logloss: 0.27877
[200]	valid_0's auc: 0.851414	valid_0's binary_logloss: 0.258698
[300]	valid_0's auc: 0.865458	valid_0's binary_logloss: 0.246029
[400]	valid_0's auc: 0.874605	valid_0's binary_logloss: 0.236919
[500]	valid_0's auc: 0.880197	valid_0's binary_logloss: 0.230351
[600]	valid_0's auc: 0.884348	valid_0's binary_logloss: 0.225157
[700]	valid_0's auc: 0.887212	valid_0's binary_logl

In [8]:

for res in results:
    print(
        f"num_leaves={res['num_leaves']}, "
        f"mean_auc={res['mean_auc']:.6f}, "
        f"std_auc={res['std_auc']:.6f}, "
        f"oof_auc={res['oof_auc']:.6f}"
    )

num_leaves=8, mean_auc=0.891707, std_auc=0.002975, oof_auc=0.891705
num_leaves=16, mean_auc=0.892819, std_auc=0.002968, oof_auc=0.892826
num_leaves=31, mean_auc=0.891554, std_auc=0.003192, oof_auc=0.891594
num_leaves=64, mean_auc=0.890005, std_auc=0.002665, oof_auc=0.890002
num_leaves=128, mean_auc=0.887842, std_auc=0.003016, oof_auc=0.887818
